# PSN-2 Training — Kaggle

**One-time setup (first session only):**
1. Upload `psn2_kaggle_full.zip` as a Kaggle Dataset → name it `psn2-kaggle`
2. Go to **Account → Settings → API → Create New Token** → download `kaggle.json`
3. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Notebook Secrets** (the 🔑 panel on the left)
4. Attach the `psn2-kaggle` dataset to this notebook
5. Set `STAGE` in the config cell and run all cells

**Every subsequent session:** just run all cells — checkpoint auto-resumes.

**How persistence works:**
- Checkpoints save to `/kaggle/working/artifacts/` every 30 min during training
- At session end (or crash), the checkpoint is pushed to a Kaggle dataset called `psn2-checkpoint`
- Next session pulls it automatically from `/kaggle/input/psn2-checkpoint/`
- No manual download/upload needed

**Session plan (PRD Section 17.3):**
- Session 1-2: D1 — ARC-AGI-2 grids + synthetic relational graphs
- Session 3-4: D2 — Causal grounding
- Session 5:   D3 — ToM/ToMi theory-of-mind
- Session 6-7: D4 — Wikitext linguistic grounding
- Session 8-9: D5 — ARC-AGI-2 + GSM8K abstract reasoning
- Session 10+: D6 — Full integration + BBH

In [ ]:
# ── 1. Secrets & Kaggle API credentials ─────────────────────────────────────
import os, sys, json, shutil, subprocess, atexit, signal, time
from pathlib import Path

# Load Kaggle credentials from Notebook Secrets
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret('KAGGLE_USERNAME')
    KAGGLE_KEY      = _secrets.get_secret('KAGGLE_KEY')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    # Write ~/.kaggle/kaggle.json so the CLI works too
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_text(
        json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY})
    )
    (kaggle_dir / 'kaggle.json').chmod(0o600)
    print(f'Kaggle credentials loaded for user: {KAGGLE_USERNAME}')
    KAGGLE_API_AVAILABLE = True
except Exception as e:
    print(f'WARNING: Kaggle secrets not found ({e})')
    print('Checkpoint auto-push disabled. Add KAGGLE_USERNAME and KAGGLE_KEY as Notebook Secrets.')
    KAGGLE_USERNAME = None
    KAGGLE_API_AVAILABLE = False

# Dataset name used to persist checkpoints across sessions
CHECKPOINT_DATASET = 'psn2-checkpoint'

In [ ]:
# ── 2. Locate repo root ──────────────────────────────────────────────────────
WORKDIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.py' in files:
        WORKDIR = root
        break

assert WORKDIR, (
    'Could not find train.py under /kaggle/input. '
    'Attach the psn2-kaggle dataset to this notebook.'
)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
print('Repo root:', WORKDIR)

In [ ]:
# ── 3. Install deps & verify GPU ─────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm', 'kaggle'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

In [ ]:
# ── 4. Checkpoint push/pull helpers ─────────────────────────────────────────
ARTIFACTS_DIR = '/kaggle/working/artifacts'
LATEST_CKPT   = f'{ARTIFACTS_DIR}/latest.pt'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

def push_checkpoint(reason='session_end'):
    """Push latest.pt to the psn2-checkpoint Kaggle dataset."""
    if not KAGGLE_API_AVAILABLE:
        print('  [push] Skipped — no Kaggle credentials')
        return
    if not os.path.exists(LATEST_CKPT):
        print('  [push] No checkpoint to push')
        return

    size_mb = os.path.getsize(LATEST_CKPT) / 1e6
    print(f'  [push] Pushing checkpoint ({size_mb:.1f} MB) — reason: {reason}')

    push_dir = '/kaggle/working/ckpt_push'
    os.makedirs(push_dir, exist_ok=True)
    shutil.copy(LATEST_CKPT, f'{push_dir}/latest.pt')

    # Write dataset-metadata.json for the Kaggle API
    meta = {
        'title': 'PSN-2 Checkpoint',
        'id': f'{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}',
        'licenses': [{'name': 'other'}],
    }
    with open(f'{push_dir}/dataset-metadata.json', 'w') as f:
        json.dump(meta, f)

    # Create dataset on first push, add new version on subsequent pushes
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', push_dir,
         '-m', f'auto-push: {reason} at {time.strftime("%Y-%m-%d %H:%M")}',
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('  [push] Done:', result.stdout.strip())
    else:
        # Dataset may not exist yet — create it
        result2 = subprocess.run(
            ['kaggle', 'datasets', 'create', '-p', push_dir, '--dir-mode', 'zip'],
            capture_output=True, text=True
        )
        if result2.returncode == 0:
            print('  [push] Created new dataset:', result2.stdout.strip())
        else:
            print('  [push] FAILED:', result.stderr.strip(), result2.stderr.strip())


def pull_checkpoint():
    """Copy latest.pt from /kaggle/input/psn2-checkpoint/ if present."""
    # Check if the checkpoint dataset was attached
    for dirpath, _, files in os.walk('/kaggle/input'):
        if 'latest.pt' in files and 'psn2-kaggle' not in dirpath:
            src = os.path.join(dirpath, 'latest.pt')
            shutil.copy(src, LATEST_CKPT)
            size_mb = os.path.getsize(LATEST_CKPT) / 1e6
            print(f'  [pull] Loaded checkpoint from {src} ({size_mb:.1f} MB)')
            return True
    print('  [pull] No previous checkpoint found — starting fresh')
    return False


# Register crash/exit handler so checkpoint is always pushed
_push_registered = False
def _on_exit():
    push_checkpoint(reason='exit_handler')

def _on_signal(signum, frame):
    push_checkpoint(reason=f'signal_{signum}')
    sys.exit(1)

if not _push_registered:
    atexit.register(_on_exit)
    for sig in (signal.SIGTERM, signal.SIGINT):
        try:
            signal.signal(sig, _on_signal)
        except (OSError, ValueError):
            pass  # can't set signals in some notebook environments
    _push_registered = True
    print('Crash/exit handler registered — checkpoint will auto-push on any exit')

In [ ]:
# ── 5. Pull checkpoint from previous session ─────────────────────────────────
if not os.path.exists(LATEST_CKPT):
    pull_checkpoint()
else:
    size_mb = os.path.getsize(LATEST_CKPT) / 1e6
    print(f'  Checkpoint already in working dir ({size_mb:.1f} MB)')

In [ ]:
# ── 6. Verify data ───────────────────────────────────────────────────────────
data_dir = os.path.join(WORKDIR, 'data')
for rel in ['d5_arc_agi2/train.jsonl', 'd3_tom/train.jsonl', 'd3_tomi/train.jsonl',
            'd5_gsm8k/train.jsonl', 'd6_bbh/test.jsonl']:
    p = os.path.join(data_dir, rel)
    exists = os.path.exists(p)
    size_mb = os.path.getsize(p) / 1e6 if exists else 0
    print(f'  {rel:35s} {"OK (" + f"{size_mb:.1f} MB)" if exists else "MISSING"}')
wiki = os.path.join(data_dir, 'd4_wikitext/train.jsonl')
print(f'  {"d4_wikitext/train.jsonl":35s} {"OK (" + f"{os.path.getsize(wiki)/1e6:.1f} MB)" if os.path.exists(wiki) else "MISSING (optional)"}')

In [ ]:
# ── 7. Session config ────────────────────────────────────────────────────────
# EDIT THIS: Choose training mode
TRAINING_MODE = 'sequential'  # 'sequential' or 'single'
START_STAGE = 'D1'            # Starting stage (for both modes)
SKIP_GATES = False            # Set True to train all stages without gate checks

STEPS_PER_STAGE = {'D1': 20000, 'D2': 20000, 'D3': 15000,
                   'D4': 25000, 'D5': 30000, 'D6': 40000}

with open(os.path.join(WORKDIR, 'configs/default.json')) as f:
    cfg = json.load(f)

cfg.update({
    'stage':                START_STAGE,
    'vsa_dim':              512,
    'max_nodes':            256,
    'grid_size':            8,      # synthetic ARC + wikitext window size
    'arc_grid_size':        30,     # real ARC-AGI-2 pad size (max 30x30)
    'grid_vocab':           10,
    'rel_vocab_size':       64,
    'batch_size':           32,
    'steps':                STEPS_PER_STAGE[START_STAGE],
    'lr_ff':                1e-4,
    'log_every':            200,
    'checkpoint_every':     1000,
    'checkpoint_dir':       ARTIFACTS_DIR,
    'data_dir':             data_dir,
    'max_wikitext_samples': 50000,
    'n_synthetic_samples':  5000,
})

cfg_dst = '/kaggle/working/session_config.json'
with open(cfg_dst, 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Training mode: {TRAINING_MODE}')
print(f'Start stage: {START_STAGE} | Batch: {cfg["batch_size"]}')
if TRAINING_MODE == 'sequential':
    print(f'Gate checks: {"DISABLED" if SKIP_GATES else "ENABLED"}')
    print(f'Will train: D1 → D2 → D3 → D4 → D5 → D6 (with gate certification)')

In [ ]:
# ── 8. Smoke test ────────────────────────────────────────────────────────────
result = subprocess.run(
    [sys.executable, 'scripts/smoke_test.py'],
    cwd=WORKDIR, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('SMOKE TEST FAILED:')
    print(result.stderr)
    raise RuntimeError('Smoke test failed')
print('Smoke test passed.')

In [ ]:
# ── 9. Training ──────────────────────────────────────────────────────────────
resume_flag = ['--resume', LATEST_CKPT] if os.path.exists(LATEST_CKPT) else []
if resume_flag:
    print(f'Resuming from step stored in {LATEST_CKPT}')
else:
    print('Starting fresh')

if TRAINING_MODE == 'sequential':
    # Sequential training: D1 → D2 → D3 → D4 → D5 → D6
    cmd = [sys.executable, 'train_sequential.py', 
           '--config', cfg_dst,
           '--start-stage', START_STAGE] + resume_flag
    if SKIP_GATES:
        cmd.append('--skip-gates')
    print('Mode: Sequential training (D1→D2→D3→D4→D5→D6)')
else:
    # Single stage training
    cmd = [sys.executable, 'train.py', '--config', cfg_dst] + resume_flag
    print(f'Mode: Single stage training ({START_STAGE})')

print('Running:', ' '.join(cmd))

try:
    result = subprocess.run(cmd, cwd=WORKDIR)
    print('Exit code:', result.returncode)
    if result.returncode != 0:
        print('Training exited with error — pushing checkpoint anyway')
        push_checkpoint(reason='training_error')
except KeyboardInterrupt:
    print('Interrupted — pushing checkpoint')
    push_checkpoint(reason='keyboard_interrupt')
    raise

In [ ]:
# ── 10. Push checkpoint ──────────────────────────────────────────────────────
# This also runs automatically on any crash/exit via the atexit handler above.
push_checkpoint(reason='training_complete')

In [ ]:
# ── 11. Evaluation ───────────────────────────────────────────────────────────
if os.path.exists(LATEST_CKPT):
    result = subprocess.run([
        sys.executable, 'evaluate.py',
        '--config', cfg_dst,
        '--checkpoint', LATEST_CKPT,
        '--output', '/kaggle/working/eval_results.json',
    ], cwd=WORKDIR)
    if os.path.exists('/kaggle/working/eval_results.json'):
        with open('/kaggle/working/eval_results.json') as f:
            print(json.dumps(json.load(f), indent=2))
else:
    print('No checkpoint — skipping eval')

In [ ]:
# ── 12. Artifacts summary ────────────────────────────────────────────────────
if os.path.exists(ARTIFACTS_DIR):
    for f in sorted(os.listdir(ARTIFACTS_DIR)):
        size_mb = os.path.getsize(os.path.join(ARTIFACTS_DIR, f)) / 1e6
        print(f'  {f:45s}  {size_mb:.1f} MB')
else:
    print('No artifacts')